# 02 – Data quality check

Before any analysis, check what is missing, odd or inconsistent in the combined dataset. This notebook only **looks** at the data; the fixes are applied in `03_clean.ipynb`.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/spend_combined.csv", dtype=str)
print(f"{len(df):,} rows, {df.shape[1]} columns\n")
print("Missing values per column:")
print(df.isna().sum().to_string())

141,930 rows, 10 columns

Missing values per column:
council               0
body_url              0
supplier              0
amount                0
pay_date              0
transaction_number    0
expense_type          0
service_area          0
project               0
source_file           0


In [2]:
a = df["amount"].str.strip()

checks = {
    "has £ sign":           a.str.contains("£", regex=False),
    "has comma":            a.str.contains(",", regex=False),
    "negative with minus":  a.str.contains("-", regex=False),
    "negative in brackets": a.str.contains("(", regex=False),
    "other characters":     a.str.contains(r"[^0-9£,.\-()\s]", regex=True),
}
for name, mask in checks.items():
    print(f"{name:22} {mask.sum():>8,}")

print("\nExamples with other characters:")
print(a[checks["other characters"]].value_counts().head(10))

has £ sign                    0
has comma                92,292
negative with minus       2,555
negative in brackets          0
other characters              0

Examples with other characters:
Series([], Name: count, dtype: int64)


In [3]:
blank = df.apply(lambda col: col.str.strip().eq("").sum())
print("Blank (empty or spaces only) per column:")
print(blank.to_string())
print("\nDistinct values per column:")
print(df.nunique().to_string())

Blank (empty or spaces only) per column:
council                    0
body_url                   0
supplier                   0
amount                     0
pay_date                   0
transaction_number         0
expense_type               0
service_area               0
project               126344
source_file                0

Distinct values per column:
council                    1
body_url                   1
supplier                4496
amount                 65227
pay_date                 515
transaction_number    119228
expense_type             220
service_area            1204
project                  363
source_file               24


In [4]:
df["amount_num"] = pd.to_numeric(df["amount"].str.replace(",", "", regex=False), errors="coerce")

print("Could not convert to a number:", df["amount_num"].isna().sum())
print("Negative amounts:", (df["amount_num"] < 0).sum())
print("Zero amounts:", (df["amount_num"] == 0).sum())
print("Positive but under £500:", ((df["amount_num"] > 0) & (df["amount_num"] < 500)).sum())
print(f"Total across all 24 months: £{df['amount_num'].sum():,.2f}\n")
print(df["amount_num"].describe().round(2).to_string())

Could not convert to a number: 0
Negative amounts: 2555
Zero amounts: 23
Positive but under £500: 16580
Total across all 24 months: £1,597,049,909.20

count      141930.00
mean        11252.38
std        142332.46
min       -632153.60
25%           678.40
50%          1700.00
75%          5558.61
max      36497535.00


In [5]:
df["pay_date_dt"] = pd.to_datetime(df["pay_date"], format="%d/%m/%Y", errors="coerce")
print("Dates that could not be read:", df["pay_date_dt"].isna().sum())
print("Earliest:", df["pay_date_dt"].min().date(), "| Latest:", df["pay_date_dt"].max().date())

# Does each payment's month match the month of the file it came from?
fm = pd.to_datetime(df["source_file"].str.extract(r"spend_(\d{4}_\d{2})")[0], format="%Y_%m")
gap = (df["pay_date_dt"].dt.year * 12 + df["pay_date_dt"].dt.month) - (fm.dt.year * 12 + fm.dt.month)
print("\nPayment month minus file month (0 = same month):")
print(gap.value_counts().sort_index().to_string())

Dates that could not be read: 0
Earliest: 2023-12-28 | Latest: 2026-03-31

Payment month minus file month (0 = same month):
-12         5
-4          1
-3          3
-2          1
-1         89
 0     141831


In [6]:
cols = ["pay_date", "supplier", "amount_num", "expense_type", "service_area"]
print("10 largest payments:")
print(df.nlargest(10, "amount_num")[cols].to_string())
print("\n10 largest credits (negative):")
print(df.nsmallest(10, "amount_num")[cols].to_string())

10 largest payments:
          pay_date                                               supplier   amount_num                          expense_type                                   service_area
83371   29/04/2025                                    BANES Pension Fund   36497535.00                 Pension Contributions                        Corporate Balance Sheet
30690   30/04/2024                                      Avon Pension Fund  30000000.00                 Pension Contributions                        Corporate Balance Sheet
64170   20/02/2025                                  Eversheds Sunderland    8491737.11     Short Term Creditors - Government                        Balance Sheet - Capital
1041    02/04/2024  Bristol, North Somerset and South Gloucestershire ICB   5651000.00  TPP - Payments to Health Authorities                          Block Placement Voids
40749   04/10/2024     South Gloucestershire Council (Main Order Account)   5391035.00                      City Region

In [7]:
by_type = (df.groupby("expense_type")["amount_num"]
             .agg(payments="count", total="sum")
             .sort_values("total", ascending=False))
by_type["total"] = by_type["total"].round(0)
by_type["share_%"] = (by_type["total"] / by_type["total"].sum() * 100).round(1)
print(by_type.head(30).to_string())

                                                       payments        total  share_%
expense_type                                                                         
New Construction                                           3526  177444500.0     11.1
Conversion, Renovation & Improvement                      10096  119712378.0      7.5
TPP - Other                                                1481   87062144.0      5.5
TPP - Placement Residential                                4038   82507632.0      5.2
Pension Contributions                                         3   70841033.0      4.4
Payments to BCC-Maintained Schools/Children’s Centres      1813   61217116.0      3.8
TPP - Grants                                               6594   55419611.0      3.5
TPP - Payments to Health Authorities                        163   55157702.0      3.5
External Professional Fees                                 2433   54274320.0      3.4
TPP - Placement Foster Care Agency                    

## Data Quality Summary

| Check | What was found | Decision |
|---|---|---|
| Missing values | No empty cells in 9 columns. `project` is blank in 126,344 of 141,930 rows (89%). | Keep `project` but don't rely on it. Fill blanks with "No project". |
| Amount format | Stored as text with commas; 2,555 negatives; no £ signs. | Remove commas and convert to numbers. |
| Total rows | 23 files ended with a total row; June 2025 had none. | Removed in notebook 01; saved in `file_totals.csv` to reconcile. |
| Dates | Day/month/year. 99 payments fall outside their file's month, 5 of them about a year earlier (earliest 28 Dec 2023). | Use the file month as `report_month`; flag mismatched rows. |
| Credits and small amounts | 2,555 credits, 23 zero-value lines, 16,580 payments under £500. Many credits are parking fine (PCN) offsets, not supplier refunds. | Keep all; add flags. |
| Very large payments | Median £1,700, mean £11,252, maximum £36.5m. | Use medians and log scales. |
| Non-supplier payments | The biggest payments are pension contributions, transfers to schools and health authorities, and balance-sheet entries. | Add `spend_group` (Procurement / Care and support / Transfers) and analyse suppliers on Procurement only. |
| Supplier names | The same payer appears under different spellings (Marston Group Limited vs Marston (Holdings) Limited). | Standardise names (case, punctuation, Ltd/Limited) into `supplier_clean`. |
| Shared transaction numbers | 119,228 distinct transaction numbers across 141,930 rows. | Several lines can share one transaction, so a shared number alone is not evidence of a duplicate. |